# 06 · Leave-one-client-out cross-domain generalization  (Phase **P6** — experiment E7)

The claim the old project made but never tested: **does the federated model generalize to a region
it never saw in training?** Here we do it properly. With **K = 5** regional clients each on a
distinct simulated sensor, we hold **one region out entirely**, train the federation on the other
four, and evaluate the **global model on the held-out region's test set** — rotating the held-out
region (leave-one-out).

Because the held-out client never trained, its BatchNorm is unknown — so we also report **AdaBN**:
re-estimate BatchNorm statistics on the held-out region's own *unlabeled* data (its test set is never
used for adaptation → no leakage). This is the label-free, deployment-realistic fix for feature shift
to a new sensor, and it ties back to the P4 finding that **BN statistics are the key lever under
feature shift**.

**Reported:** per held-out region, and averaged — `FedAvg`, `FedAvg+AdaBN`, `FedProx`,
`FedProx+AdaBN` accuracy on the unseen region. **Resumable** per `(held_out, method, seed)`.

> Sensor shift is ON (that's what makes the held-out region a genuinely *novel domain*).
> ~10 trainings at one seed (K=5, leave-one-out × 2 methods) — a couple of hours on a T4, resumable.


## 1 · Environment + Drive

In [ ]:
import os, sys, subprocess

REPO_URL = "https://github.com/sgogoigh/Satellite-Image-Classification-Fed-Learning.git"
REPO_DIR = "/content/Satellite-Image-Classification-Fed-Learning"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)
os.chdir(REPO_DIR)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

sys.path.insert(0, os.path.join(REPO_DIR, "src"))
import torch, fedsat
from fedsat.utils import get_device
print("fedsat", fedsat.__version__, "| torch", torch.__version__, "| device:", get_device())
if not torch.cuda.is_available():
    print("WARNING: no GPU. Runtime > Change runtime type > T4 GPU, then re-run this cell.")


In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = "/content/drive/MyDrive/fedsat"
except Exception as e:
    print("Drive not mounted (", e, ") -> falling back to local ./artifacts")
    DRIVE = os.path.join(REPO_DIR, "artifacts")
os.makedirs(DRIVE, exist_ok=True)
print("Artifacts dir:", DRIVE)


## 2 · Config

In [ ]:
from fedsat.config import ExperimentConfig
from fedsat.utils import get_device

BASE = ExperimentConfig(
    experiment_name="p6_loco",
    dataset="eurosat_rgb", hf_repo="blanchon/EuroSAT_RGB",
    num_clients=5, input_size=64,                 # 5 "regions" -> leave-one-region-out
    backbone="resnet18", pretrained=True, norm="bn",
    optimizer="sgd", lr=0.01, momentum=0.9, weight_decay=5e-4, batch_size=64,
    local_epochs=1, num_workers=2, max_epochs=40, early_stop_patience=7,
    data_cache_dir="/content/hf_cache",           # LOCAL disk (Drive-mounted HF cache stalls on writes)
    partition_dir=f"{DRIVE}/partitions",          # partitions/results stay on Drive (tiny, persistent)
    results_dir=f"{DRIVE}/results",
    device=get_device(),
)

ALPHA = 0.5
SHIFT_ON = True                # held-out region = a novel sensor (the point of cross-domain eval)
SHIFT_STRENGTH = 1.0
SHIFT_SEED = 0
SEEDS = [42]                   # add 43,44 for CIs
NUM_ROUNDS = 15
LOCAL_EPOCHS = 1
METHODS = [("fedavg", 0.0), ("fedprox", 0.1)]
RESULTS_DIR = f"{DRIVE}/results/p6_loco"
print(f"LOCO over K={BASE.num_clients} regions x {len(METHODS)} methods x {len(SEEDS)} seeds "
      f"(shift {'ON' if SHIFT_ON else 'OFF'})")


## 3 · Load EuroSAT + build the K=5 partition and per-region sensors

In [ ]:
from fedsat.data import (load_eurosat, integrity_gate, build_partition,
                         save_partition, load_partition, build_client_shifts)

hf_ds, class_names, labels = load_eurosat(BASE.hf_repo, cache_dir=BASE.data_cache_dir)
stats = integrity_gate(class_names, labels, expected_classes=BASE.num_classes,
                       expected_total=BASE.expected_total)
print("data ok:", stats["total"], "images |", stats["num_classes"], "classes")


## 4 · Resumable results store

In [ ]:
import os, json
os.makedirs(RESULTS_DIR, exist_ok=True)
STORE = os.path.join(RESULTS_DIR, "results.jsonl")

def load_rows():
    if not os.path.exists(STORE):
        return []
    return [json.loads(l) for l in open(STORE) if l.strip()]

def is_done(held_out, method, seed):
    return any(r.get("held_out") == held_out and r.get("method") == method and r.get("seed") == seed
               for r in load_rows())

def append_row(row):
    with open(STORE, "a") as f:
        f.write(json.dumps(row) + "\n")

print("results store:", STORE, "| existing rows:", len(load_rows()))


## 5 · The LOCO sweep

For each held-out region and each method, `run_loco` trains a global model on the other four regions
and evaluates it on the unseen region — with and without AdaBN. Resumes on re-run.


In [ ]:
from fedsat.fl import run_loco
from fedsat.utils import set_seed
import time

for seed in SEEDS:
    cfg = BASE.replace(alpha=ALPHA, seed=seed)
    try:
        part = load_partition(cfg)
    except FileNotFoundError:
        part = build_partition(cfg, labels, class_names, data_hash=stats["data_hash"])
        save_partition(cfg, part)
    shifts = build_client_shifts(BASE.num_clients, seed=SHIFT_SEED, strength=SHIFT_STRENGTH) if SHIFT_ON else {}
    client_ids = sorted(part["clients"], key=int)

    for held_out in client_ids:
        for method, mu in METHODS:
            if is_done(held_out, method, seed):
                continue
            set_seed(seed)
            print(f"[held_out={held_out} seed={seed}] {method} (train on {len(client_ids)-1} regions) ...", flush=True)
            t = time.time()
            r = run_loco(cfg, hf_ds, part, class_names, held_out=held_out,
                         num_rounds=NUM_ROUNDS, local_epochs=LOCAL_EPOCHS, mu=mu,
                         client_transforms=shifts, adabn=True)
            append_row({"held_out": held_out, "method": method, "seed": seed,
                        "base_acc": r["held_out_test_acc"], "adabn_acc": r["held_out_test_acc_adabn"]})
            print(f"    unseen-region acc: base={r['held_out_test_acc']:.4f}  "
                  f"+AdaBN={r['held_out_test_acc_adabn']:.4f}  ({time.time()-t:.0f}s)", flush=True)

print("\nLOCO COMPLETE. rows:", len(load_rows()))


## 6 · Results — generalization to an unseen region

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt

df = pd.DataFrame(load_rows())

conds = []
for method, _ in METHODS:
    sub = df[df["method"] == method]
    conds.append((method, sub["base_acc"].mean(), sub["base_acc"].std(), sub["base_acc"].min()))
    conds.append((method + "+adabn", sub["adabn_acc"].mean(), sub["adabn_acc"].std(), sub["adabn_acc"].min()))
res = pd.DataFrame(conds, columns=["condition", "mean_unseen_acc", "std_over_regions", "worst_region"])
print(res.to_string(index=False))

x = np.arange(len(res))
plt.figure(figsize=(9, 5))
plt.bar(x, res["mean_unseen_acc"], yerr=res["std_over_regions"], capsize=4,
        color=["#d35400", "#e67e22", "#8e44ad", "#9b59b6"][:len(res)])
plt.xticks(x, res["condition"], rotation=15, ha="right")
plt.ylabel("accuracy on the UNSEEN region"); plt.ylim(0, 1)
plt.grid(axis="y", alpha=0.3)
plt.title(f"Leave-one-region-out generalization (K={BASE.num_clients}, sensor shift {'ON' if SHIFT_ON else 'OFF'})")
plt.tight_layout(); plt.show()


## 7 · Per-region breakdown + verdict + save

In [ ]:
piv = df.pivot_table(index="held_out", columns="method", values=["base_acc", "adabn_acc"])
print("per held-out region (rows) — base vs +AdaBN by method:")
print(piv.to_string())

fa = df[df["method"] == "fedavg"]
base_mean = fa["base_acc"].mean(); ada_mean = fa["adabn_acc"].mean()
print(f"\nFedAvg global on unseen region: base {base_mean:.4f}  ->  +AdaBN {ada_mean:.4f}  "
      f"(AdaBN delta {ada_mean-base_mean:+.4f})")
print("Interpretation: a global model degrades on a NOVEL sensor; AdaBN (label-free BN re-estimation")
print("on the unseen region's own data) recovers much of the gap -- BN statistics are the key lever.")

df.to_csv(os.path.join(RESULTS_DIR, "p6_summary.csv"), index=False)
res.to_csv(os.path.join(RESULTS_DIR, "p6_conditions.csv"), index=False)
print("\nsaved ->", RESULTS_DIR)


## Done — P6 complete

This is the honest cross-domain-generalization result the original project claimed but never
produced: the federated global model's accuracy on a **region it never trained on**, and how much a
cheap, **label-free AdaBN** adaptation recovers under sensor shift. Together with P4 (participating
clients → FedBN) this gives a complete BN-centric story for feature shift: personalized BN for
regions in the federation, AdaBN for regions outside it.

**Next — Phase P7 (`07_analysis_figures.ipynb`):** pull every phase's saved results into one master
table (mean ± CI, paired significance tests), regenerate the publication figures (α-curve, FedBN
ablation, scale/comm, LOCO), and rewrite report §5–§8 from these real numbers.

**Optional extensions (say the word):** multispectral EuroSAT_MSI (13-band) RGB-vs-MS ablation, and
the Track-B genuine multi-dataset cross-domain study (EuroSAT + AID + NWPU + UC-Merced).
